# Feature 1A: Acoustic Tracking with Speech Envelope

This notebook builds the first subject-level acoustic tracking feature using the broadband `speech_envelope` predictor.

Core idea: if a subject's EEG is more predictable from the speech envelope, that subject has stronger acoustic tracking.

## Method note

This first pass does not require TRF-Tools or Eelbrain. It uses a transparent ridge encoding model with lagged speech-envelope predictors:

```text
lagged speech envelope -> EEG channels
```

TRF-Tools/Eelbrain can be useful later for a more standardized TRF workflow, but a small self-contained ridge model is easier to inspect and debug for the first feature.

## Outputs

Subject-level feature table:

```text
/Users/yanyuwoo/Data/Alice Comprehension/features/acoustic_tracking_speech_envelope.csv
```

Fold/channel-level detail table:

```text
/Users/yanyuwoo/Data/Alice Comprehension/intermediate/acoustic_tracking_speech_envelope/fold_channel_scores.csv
```

QC table:

```text
/Users/yanyuwoo/Data/Alice Comprehension/qc/acoustic_tracking_speech_envelope_qc.csv
```

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import mne

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
ANALYSIS_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')

PREDICTOR_DIR = ANALYSIS_ROOT / 'predictors' / 'speech_envelope'
FEATURE_DIR = ANALYSIS_ROOT / 'features'
QC_DIR = ANALYSIS_ROOT / 'qc'
INTERMEDIATE_DIR = ANALYSIS_ROOT / 'intermediate' / 'acoustic_tracking_speech_envelope'

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

EEG_FS = 100
EEG_FILTER_L_FREQ = 1.0
EEG_FILTER_H_FREQ = 8.0

LAG_START_MS = 0
LAG_END_MS = 500
RIDGE_ALPHA = 100.0

MIN_SEGMENTS_PER_SUBJECT = 8

mne.set_log_level('WARNING')

## Helper functions

In [2]:
def subject_dirs():
    return sorted(
        [p for p in BIDS_ROOT.glob('sub-*') if p.is_dir()],
        key=lambda p: int(p.name.split('-')[1]),
    )


def parse_stimulus_id(trial_type):
    if not isinstance(trial_type, str) or not trial_type.startswith('Stimulus/'):
        return None
    return int(trial_type.split('/')[-1])


def load_events(subject):
    path = BIDS_ROOT / subject / 'eeg' / f'{subject}_task-alice_events.tsv'
    events = pd.read_csv(path, sep='\t', encoding='utf-8-sig')

    # Prefer the normalized BIDS column created by the events-unification notebook.
    # Fall back to parsing trial_type only for older events files.
    if 'stimulus_id' in events.columns:
        events['stimulus_id'] = pd.to_numeric(events['stimulus_id'], errors='coerce')
    else:
        events['stimulus_id'] = events['trial_type'].map(parse_stimulus_id)

    events = events.dropna(subset=['stimulus_id']).copy()
    events['stimulus_id'] = events['stimulus_id'].astype(int)
    return events.sort_values('onset')


def load_speech_envelope(stimulus_id):
    path = PREDICTOR_DIR / f'stim-{stimulus_id:02d}_speech_envelope_fs-{EEG_FS}.npz'
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as npz:
        data = npz['data'].astype(np.float64)
        metadata = json.loads(str(npz['metadata']))
    if data.ndim != 2 or data.shape[1] != 1:
        raise ValueError(f'Unexpected predictor shape for {path}: {data.shape}')
    return data, metadata


AUDIO_LIKE_EXACT = {'aud', 'audio', 'aux', 'aux5', 'ox'}
AUDIO_LIKE_PATTERN = re.compile(r'(aud|audio|aux|stim|trigger|trig)', re.IGNORECASE)


def is_audio_like_channel(ch_name):
    lower = ch_name.strip().lower()
    return lower in AUDIO_LIKE_EXACT or AUDIO_LIKE_PATTERN.search(lower) is not None


def mark_audio_like_channels_as_misc(raw):
    audio_like = [ch for ch in raw.ch_names if is_audio_like_channel(ch)]
    if audio_like:
        raw.set_channel_types({ch: 'misc' for ch in audio_like})
    return audio_like


def load_subject_eeg(subject):
    vhdr_path = BIDS_ROOT / subject / 'eeg' / f'{subject}_task-alice_eeg.vhdr'
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)
    excluded_non_neural_channels = mark_audio_like_channels_as_misc(raw)
    raw.pick('eeg')
    raw.filter(EEG_FILTER_L_FREQ, EEG_FILTER_H_FREQ, phase='zero-double')
    raw.resample(EEG_FS)
    return raw, excluded_non_neural_channels


def crop_eeg_segment(raw, onset_sec, n_samples):
    start = int(round(onset_sec * raw.info['sfreq']))
    stop = start + n_samples
    data = raw.get_data(start=start, stop=stop).T
    if data.shape[0] != n_samples:
        return None
    return data


def lag_samples():
    return np.arange(
        int(round(LAG_START_MS / 1000 * EEG_FS)),
        int(round(LAG_END_MS / 1000 * EEG_FS)) + 1,
    )


def make_lagged_design(x, lags):
    if x.ndim == 1:
        x = x[:, None]
    max_lag = int(lags.max())
    rows = []
    for lag in lags:
        rows.append(x[max_lag - lag : x.shape[0] - lag])
    return np.hstack(rows)


def align_y_to_lags(y, lags):
    max_lag = int(lags.max())
    return y[max_lag:]


def standardize_train_test(x_train, x_test):
    mu = x_train.mean(axis=0, keepdims=True)
    sd = x_train.std(axis=0, keepdims=True)
    sd[sd == 0] = 1.0
    return (x_train - mu) / sd, (x_test - mu) / sd


def ridge_fit_predict(x_train, y_train, x_test, alpha):
    x_train, x_test = standardize_train_test(x_train, x_test)
    y_mu = y_train.mean(axis=0, keepdims=True)
    y_train_centered = y_train - y_mu

    penalty = alpha * np.eye(x_train.shape[1])
    beta = np.linalg.solve(x_train.T @ x_train + penalty, x_train.T @ y_train_centered)
    return x_test @ beta + y_mu


def channel_correlations(y_true, y_pred):
    y_true = y_true - y_true.mean(axis=0, keepdims=True)
    y_pred = y_pred - y_pred.mean(axis=0, keepdims=True)
    denom = np.sqrt((y_true ** 2).sum(axis=0) * (y_pred ** 2).sum(axis=0))
    with np.errstate(divide='ignore', invalid='ignore'):
        r = (y_true * y_pred).sum(axis=0) / denom
    return r

## Parser sanity check

Run this before the subject loop. It confirms that both event-marker formats are supported in the current kernel.

## Build acoustic tracking feature

This cell does the full subject loop. It uses leave-one-stimulus-segment-out cross-validation.

In [4]:
feature_rows = []
qc_rows = []
fold_channel_rows = []
lags = lag_samples()

for sub_dir in subject_dirs():
    subject = sub_dir.name
    try:
        events = load_events(subject)
        raw, excluded_non_neural_channels = load_subject_eeg(subject)
        channel_names = raw.ch_names

        segments = []
        missing_segments = []

        for _, event in events.iterrows():
            stimulus_id = int(event['stimulus_id'])
            predictor, predictor_meta = load_speech_envelope(stimulus_id)
            eeg = crop_eeg_segment(raw, float(event['onset']), predictor.shape[0])

            if eeg is None:
                missing_segments.append(stimulus_id)
                continue

            x_lagged = make_lagged_design(predictor, lags)
            y_aligned = align_y_to_lags(eeg, lags)
            segments.append({
                'stimulus_id': stimulus_id,
                'x': x_lagged,
                'y': y_aligned,
            })

        if len(segments) < MIN_SEGMENTS_PER_SUBJECT:
            raise ValueError(f'Only {len(segments)} usable segments')

        fold_rs = []

        for test_idx, test_segment in enumerate(segments):
            train_segments = [seg for i, seg in enumerate(segments) if i != test_idx]
            x_train = np.vstack([seg['x'] for seg in train_segments])
            y_train = np.vstack([seg['y'] for seg in train_segments])
            x_test = test_segment['x']
            y_test = test_segment['y']

            y_pred = ridge_fit_predict(x_train, y_train, x_test, RIDGE_ALPHA)
            r = channel_correlations(y_test, y_pred)
            fold_rs.append(r)

            for ch_name, ch_r in zip(channel_names, r):
                fold_channel_rows.append({
                    'subject': subject,
                    'predictor_name': 'speech_envelope',
                    'test_stimulus_id': test_segment['stimulus_id'],
                    'channel': ch_name,
                    'tracking_r': float(ch_r) if np.isfinite(ch_r) else np.nan,
                })

        fold_r_matrix = np.vstack(fold_rs)
        channel_mean_r = np.nanmean(fold_r_matrix, axis=0)
        best_channel_idx = int(np.nanargmax(channel_mean_r))
        best_channel_name = channel_names[best_channel_idx]

        feature_rows.append({
            'subject': subject,
            'predictor_name': 'speech_envelope',
            'tracking_r_mean': float(np.nanmean(fold_r_matrix)),
            'tracking_r_median': float(np.nanmedian(fold_r_matrix)),
            'tracking_r_best_channel': float(channel_mean_r[best_channel_idx]),
            'tracking_r_best_channel_name': best_channel_name,
            'tracking_r_channel_mean_sd': float(np.nanstd(channel_mean_r)),
            'n_segments_used': len(segments),
            'n_channels_used': len(channel_names),
            'excluded_non_neural_channels': ';'.join(excluded_non_neural_channels),
            'lag_start_ms': LAG_START_MS,
            'lag_end_ms': LAG_END_MS,
            'ridge_alpha': RIDGE_ALPHA,
            'qc_flag': 'ok',
        })

        qc_rows.append({
            'subject': subject,
            'n_events': len(events),
            'n_segments_used': len(segments),
            'n_missing_segments': len(missing_segments),
            'missing_segments': ';'.join(map(str, missing_segments)),
            'n_channels_used': len(channel_names),
            'excluded_non_neural_channels': ';'.join(excluded_non_neural_channels),
            'raw_duration_sec': raw.times[-1],
            'qc_flag': 'ok',
        })

    except Exception as exc:
        warnings.warn(f'{subject} failed: {exc}')
        qc_rows.append({
            'subject': subject,
            'n_events': np.nan,
            'n_segments_used': 0,
            'n_missing_segments': np.nan,
            'missing_segments': '',
            'n_channels_used': np.nan,
            'excluded_non_neural_channels': '',
            'raw_duration_sec': np.nan,
            'qc_flag': f'failed: {exc}',
        })

features = pd.DataFrame(feature_rows).sort_values('subject')
qc = pd.DataFrame(qc_rows).sort_values('subject')
fold_channel_scores = pd.DataFrame(fold_channel_rows)

features.to_csv(FEATURE_DIR / 'acoustic_tracking_speech_envelope.csv', index=False)
qc.to_csv(QC_DIR / 'acoustic_tracking_speech_envelope_qc.csv', index=False)
fold_channel_scores.to_csv(INTERMEDIATE_DIR / 'fold_channel_scores.csv', index=False)

features

,subject,predictor_name,tracking_r_mean,tracking_r_median,tracking_r_best_channel,tracking_r_channel_mean_sd,n_segments_used,n_channels_used,lag_start_ms,lag_end_ms,ridge_alpha,qc_flag
0,sub-01,speech_envelope,0.052133,0.036524,0.844025,0.105962,12,62,0,500,100.0,ok
1,sub-02,speech_envelope,0.039619,0.027323,0.928536,0.115973,11,62,0,500,100.0,ok
2,sub-03,speech_envelope,0.049095,0.030125,0.868764,0.109835,12,62,0,500,100.0,ok
3,sub-04,speech_envelope,0.028504,0.015159,0.917945,0.115335,12,62,0,500,100.0,ok
4,sub-05,speech_envelope,0.015002,0.022516,0.037555,0.011449,12,61,0,500,100.0,ok
5,sub-06,speech_envelope,0.052515,0.035302,0.892042,0.111427,12,62,0,500,100.0,ok
6,sub-07,speech_envelope,0.035555,0.016880,0.930348,0.115853,12,62,0,500,100.0,ok
7,sub-08,speech_envelope,0.027681,0.011529,0.915143,0.116253,12,62,0,500,100.0,ok
8,sub-09,speech_envelope,0.043978,0.046537,0.072587,0.022246,12,62,0,500,100.0,ok
9,sub-10,speech_envelope,0.037124,0.017744,0.929161,0.115974,12,62,0,500,100.0,ok


## Quick merge with comprehension scores

Optional: run this after feature generation to inspect the behavioral association.

In [5]:
def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'derived' / 'comprehension_scores_clean.csv').exists():
            return path
    raise FileNotFoundError('Could not find project root with data/derived/comprehension_scores_clean.csv')


project_root = find_project_root()
comprehension_path = project_root / 'data' / 'derived' / 'comprehension_scores_clean.csv'
feature_path = FEATURE_DIR / 'acoustic_tracking_speech_envelope.csv'

print(f'Project root: {project_root}')
print(f'Comprehension path exists: {comprehension_path.exists()}')
print(f'Feature path exists: {feature_path.exists()}')

if 'features' not in globals():
    features = pd.read_csv(feature_path)

comprehension = pd.read_csv(comprehension_path)
comprehension_for_merge = comprehension.rename(columns={'participant_id': 'subject'})
merged = comprehension_for_merge.merge(features, on='subject', how='inner')

print(f'Merged rows: {len(merged)}')
display(merged[['subject', 'score_prop', 'tracking_r_mean', 'tracking_r_best_channel']])
print(merged[['score_prop', 'tracking_r_mean', 'tracking_r_best_channel']].corr())

Project root: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction
Comprehension path exists: True
Feature path exists: True
Merged rows: 49


,subject,score_prop,tracking_r_mean,tracking_r_best_channel
0,sub-01,0.750,0.052133,0.844025
1,sub-02,0.750,0.039619,0.928536
2,sub-03,0.750,0.049095,0.868764
3,sub-04,0.875,0.028504,0.917945
4,sub-05,0.875,0.015002,0.037555
5,sub-06,0.625,0.052515,0.892042
6,sub-07,0.500,0.035555,0.930348
7,sub-08,0.750,0.027681,0.915143
8,sub-09,0.250,0.043978,0.072587
9,sub-10,0.625,0.037124,0.929161


                         score_prop  tracking_r_mean  tracking_r_best_channel
score_prop                 1.000000         0.233260                 0.082491
tracking_r_mean            0.233260         1.000000                 0.281275
tracking_r_best_channel    0.082491         0.281275                 1.000000
